# Cleaning Notebook

This notebook loads the raw SMS export and creates a cleaned dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import re

base_dir = Path.cwd().parent
raw_path = base_dir / '1_Data_Collection' / 'raw_data.csv'
cleaned_path = base_dir / '2_Data_Cleaning' / 'cleaned_data.csv'
quality_path = base_dir / '2_Data_Cleaning' / 'data_quality_report.csv'

raw = pd.read_csv(raw_path)
raw['content'] = raw['content'].astype(str)

failed_re = re.compile(r'failed|rejected|echec|echoue|annule|canceled|cancelled', re.IGNORECASE)
airtime_re = re.compile(r'\bairtime\b', re.IGNORECASE)
balance_re = re.compile(r'\b(solde|balance)\b', re.IGNORECASE)
adjustment_re = re.compile(r'\badjustment|ajustement\b', re.IGNORECASE)
otp_re = re.compile(r'\b(code|otp|login|verifier|verification)\b', re.IGNORECASE)
promo_re = re.compile(r'\b(bonus|promo|promotion|cadeau|win|gagner|scholarship)\b', re.IGNORECASE)

raw['status'] = raw['content'].apply(lambda text: 'failed' if failed_re.search(str(text)) else 'success')

raw['transaction_type'] = raw['transaction_type'].astype(str).str.lower()
raw.loc[raw['content'].str.contains(airtime_re, na=False), 'transaction_type'] = 'airtime'
raw.loc[raw['content'].str.contains(balance_re, na=False), 'transaction_type'] = 'balance'
raw.loc[raw['content'].str.contains(adjustment_re, na=False), 'transaction_type'] = 'adjustment'

otp_mask = raw['content'].str.contains(otp_re, na=False)
promo_mask = raw['content'].str.contains(promo_re, na=False)
cleaned = raw[~otp_mask & ~promo_mask].copy()

cleaned = cleaned.dropna(subset=['datetime', 'content'])
cleaned = cleaned.drop_duplicates(subset=['user_id', 'datetime', 'content', 'amount', 'transaction_type'])

keep_types = {'receive', 'withdraw', 'transfer', 'payment', 'airtime', 'balance', 'adjustment'}
cleaned = cleaned[cleaned['transaction_type'].isin(keep_types)]

cleaned.loc[(cleaned['transaction_type'] == 'balance') & (cleaned['amount'].isna()), 'amount'] = 0
cleaned = cleaned[cleaned['amount'].fillna(0) >= 0]
cleaned = cleaned[~cleaned['amount'].isna() | (cleaned['transaction_type'] == 'balance')]

cleaned.to_csv(cleaned_path, index=False)
quality = cleaned.isna().mean().reset_index()
quality.columns = ['column', 'missing_rate']
quality.to_csv(quality_path, index=False)

display(cleaned.head())
display(quality.head())